In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

def build_crime_feature_file(
    file_path,
    output_path=None,
    time_freq="h",
    use_full_day_grid=True
):
    """
    Build a processed feature table from raw incident-level crime data.

    Input:
        Raw CSV with at least:
        - Date
        - Community Area

    Output:
        One row per (Community Area, hour_slot), with engineered features.
    """

    # 1. Load only needed columns
    df = pd.read_csv(file_path, usecols=["Date", "Community Area"])

    # 2. Parse and clean
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Community Area"] = pd.to_numeric(df["Community Area"], errors="coerce")

    df = df.dropna(subset=["Date", "Community Area"]).copy()
    df["Community Area"] = df["Community Area"].astype(int)

    # 3. Create time slot
    df["hour_slot"] = df["Date"].dt.floor(time_freq)

    # 4. Aggregate to Community Area + hour
    crime_counts = (
        df.groupby(["Community Area", "hour_slot"])
          .size()
          .reset_index(name="crime_count")
    )
    crime_counts["crime_occurrence"] = (crime_counts["crime_count"] > 0).astype(int)

    # 5. Create full grid
    all_areas = sorted(df["Community Area"].unique())

    if use_full_day_grid:
        start_day = df["Date"].dt.floor("D").min()
        end_day = df["Date"].dt.floor("D").max()

        all_hours = pd.date_range(
            start=start_day,
            end=end_day + pd.Timedelta(hours=23),
            freq=time_freq
        )
    else:
        all_hours = pd.date_range(
            start=df["hour_slot"].min(),
            end=df["hour_slot"].max(),
            freq=time_freq
        )

    full_grid = pd.MultiIndex.from_product(
        [all_areas, all_hours],
        names=["Community Area", "hour_slot"]
    ).to_frame(index=False)

    # 6. Merge counts into full grid
    data = full_grid.merge(
        crime_counts,
        on=["Community Area", "hour_slot"],
        how="left"
    )

    data["crime_count"] = data["crime_count"].fillna(0)
    data["crime_occurrence"] = data["crime_occurrence"].fillna(0).astype(int)

    # 7. Time features
    data["hour"] = data["hour_slot"].dt.hour
    data["day_of_week"] = data["hour_slot"].dt.dayofweek
    data["month"] = data["hour_slot"].dt.month
    data["day"] = data["hour_slot"].dt.day
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

    # 8. Cyclical hour encoding
    data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
    data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

    # 9. Sort before lag features
    data = data.sort_values(["Community Area", "hour_slot"]).reset_index(drop=True)

    # 10. Lag features based on past crime_count
    grouped = data.groupby("Community Area")["crime_count"]

    data["lag_1h"] = grouped.shift(1)
    data["lag_2h"] = grouped.shift(2)
    data["lag_3h"] = grouped.shift(3)
    data["lag_24h"] = grouped.shift(24)

    # 11. Rolling features using only past values
    shifted = grouped.shift(1)

    data["rolling_3h_mean"] = shifted.rolling(3).mean().reset_index(level=0, drop=True)
    data["rolling_6h_mean"] = shifted.rolling(6).mean().reset_index(level=0, drop=True)
    data["rolling_12h_mean"] = shifted.rolling(12).mean().reset_index(level=0, drop=True)
    data["rolling_24h_mean"] = shifted.rolling(24).mean().reset_index(level=0, drop=True)

    # 12. Fill missing lag/rolling values
    lag_roll_cols = [
        "lag_1h", "lag_2h", "lag_3h", "lag_24h",
        "rolling_3h_mean", "rolling_6h_mean",
        "rolling_12h_mean", "rolling_24h_mean"
    ]
    data[lag_roll_cols] = data[lag_roll_cols].fillna(0)

    # 13. Type cleanup
    int_cols = [
        "Community Area", "crime_occurrence",
        "hour", "day_of_week", "month", "day", "is_weekend"
    ]
    for col in int_cols:
        data[col] = data[col].astype(int)

    # 14. Save if requested
    if output_path is not None:
        data.to_csv(output_path, index=False)

    return data

In [ ]:
processed_df = build_crime_feature_file(
    file_path="/content/drive/MyDrive/Chicago_Crimes_SEIS764.csv"
)

In [ ]:
processed_df = processed_df.sort_values(by="hour_slot").reset_index(drop=True)

In [ ]:
processed_df

In [ ]:
processed_df.columns

In [ ]:
# target column
target_col = "crime_occurrence"

# feature columns
feature_cols = [
    "Community Area",
    "day_of_week",
    "month",
    "day",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "lag_1h",
    "lag_2h",
    "lag_3h",
    "lag_24h",
    "rolling_3h_mean",
    "rolling_6h_mean",
    "rolling_12h_mean",
    "rolling_24h_mean"
]

# keep only required columns
model_df = processed_df[feature_cols + [target_col]].copy()

# separate features and target
X = model_df[feature_cols]
y = model_df[target_col]

# train-test split
# shuffle=False keeps chronological order
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# check shapes
print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
X

In [ ]:

from tensorflow.keras.optimizers import Adam

model0 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(10, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Define optimizer with learning rate
optimizer = Adam(learning_rate=0.001)

model0.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model0.summary()

history = model0.fit(X_train_scaled,y_train,validation_split=0.2,epochs=50,batch_size=128,verbose=1, validation_data=(X_test_scaled, y_test))


In [ ]:
# Predict probabilities
y_prob = model0.predict(X_test_scaled).ravel()

# Convert probabilities to class labels
y_pred = (y_prob >= 0.5).astype(int)

# Evaluation metrics
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from tensorflow.keras.optimizers import Adam

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

model1 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(10, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Define optimizer with learning rate
optimizer = Adam(learning_rate=0.001)

model1.compile(optimizer=optimizer,loss="binary_crossentropy",metrics=["accuracy"]
)

model1.summary()

history1 = model1.fit(X_train_scaled,y_train,validation_split=0.2,epochs=30,batch_size=128,verbose=1,callbacks=[early_stop], validation_data=(X_test_scaled, y_test))

In [ ]:
# Predict probabilities
y_prob1 = model1.predict(X_test_scaled).ravel()

# Convert probabilities to class labels
y_pred = (y_prob1 >= 0.5).astype(int)

# Evaluation metrics
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob1))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from tensorflow.keras.optimizers import SGD

model2 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(10, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Define SGD optimizer with learning rate
optimizer = SGD(learning_rate=0.01, momentum=0.9)

model2.compile(optimizer=optimizer,loss="binary_crossentropy",metrics=["accuracy"])

model2.summary()

history2 = model2.fit(X_train_scaled,y_train,validation_data=(X_test_scaled, y_test),epochs=30,batch_size=128,verbose=1, callbacks=[early_stop],)

In [ ]:
# Predict probabilities
y_prob2 = model2.predict(X_test_scaled).ravel()

# Convert probabilities to class labels
y_pred = (y_prob2 >= 0.5).astype(int)

# Evaluation metrics
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob2))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Original training data shape:", X_train_scaled.shape, y_train.shape)
print("Resampled training data shape:", X_train_resampled.shape, y_train_resampled.shape)

# Check the class distribution after SMOTE
unique_classes, class_counts = np.unique(y_train_resampled, return_counts=True)
print("Class distribution after SMOTE:", dict(zip(unique_classes, class_counts)))

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import Sequential # Import Sequential

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Define the neural network model
model3 = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),  # Input layer matching the number of feature
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')  # Output layer for binary classification with sigmoid activation
])

# Compile the model
optimizer = Adam(learning_rate=0.0005)
model3.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
model3.summary()

# Train the model
history3 = model3.fit(X_train_resampled, y_train_resampled, epochs=10, batch_size=128, verbose=1, validation_data=(X_test_scaled, y_test))

print("\nNeural network training complete.")

In [ ]:
# Predict probabilities
y_prob3 = model3.predict(X_test_scaled).ravel()

# Convert probabilities to class labels
y_pred = (y_prob3 >= 0.5).astype(int)

# Evaluation metrics
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import Sequential # Import Sequential

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Define the neural network model
model4 = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),  # Input layer matching the number of feature
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')  # Output layer for binary classification with sigmoid activation
])

# Compile the model
optimizer = Adam(learning_rate=0.0005)
model4.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Diplay the model summary
model4.summary()

# Train the model
history4 = model4.fit(X_train_resampled, y_train_resampled, epochs=20, batch_size=128, verbose=1, callbacks=[early_stop],validation_data=(X_test_scaled, y_test))

print("\nNeural network training complete.")

In [ ]:
# Predict probabilities
y_prob4 = model4.predict(X_test_scaled).ravel()

# Convert probabilities to class labels
y_pred = (y_prob3 >= 0.5).astype(int)

# Evaluation metrics
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import accuracy_score
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)
print(random_search.best_score_)

### Model 5: Deeper Network with Adjusted Dropout and Lower Learning Rate (with SMOTE and Early Stopping)

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Original training data shape:", X_train_scaled.shape, y_train.shape)
print("Resampled training data shape:", X_train_resampled.shape, y_train_resampled.shape)

# Check the class distribution after SMOTE
unique_classes, class_counts = np.unique(y_train_resampled, return_counts=True)
print("Class distribution after SMOTE:", dict(zip(unique_classes, class_counts)))

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import Sequential

# Re-define EarlyStopping for clarity, though it's already defined
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5, # Increased patience for this model
    restore_best_weights=True
)

# Define the neural network model with a deeper structure and different dropout
model5 = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),  # Input layer matching the number of features
    Dense(256, activation='relu'),
    Dropout(0.5), # Higher dropout rate
    Dense(128, activation='relu'),
    Dropout(0.3), # Adjusted dropout rate
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model with a lower learning rate
optimizer = Adam(learning_rate=0.0001)
model5.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
model5.summary()

# Train the model
history5 = model5.fit(
    X_train_resampled, y_train_resampled,
    epochs=30, # More epochs than model3/model4
    batch_size=128,
    verbose=1,
    callbacks=[early_stop],
    validation_data=(X_test_scaled, y_test)
)

print("\nNeural network training complete for model5.")

### Evaluate Model 5 Performance

In [ ]:
# Predict probabilities for model5
y_prob5 = model5.predict(X_test_scaled).ravel()

# Convert probabilities to class labels for model5
y_pred5 = (y_prob5 >= 0.5).astype(int)

# Evaluation metrics for model5
print("Classification Report for Model 5:")
print(classification_report(y_test, y_pred5, zero_division=0))
print("ROC-AUC for Model 5:", roc_auc_score(y_test, y_prob5))
print("Confusion Matrix for Model 5:\n", confusion_matrix(y_test, y_pred5))

Model 5 showed a moderate accuracy with the highest recall for crime occurrences (Class 1) but lower precision, indicating a notable number of false positives. Its F1-Score for crime occurrence was 0.51

### Demonstrating Crime Occurrence Prediction with Neural Network Model5

In [ ]:
# Select a few samples from the scaled test set for demonstration
sample_indices = [0, 1, 2, 3, 4] # Using the first 5 samples from X_test_scaled
sample_features = X_test_scaled[sample_indices]

# Make predictions (probabilities) using model5
sample_probabilities_model5 = model5.predict(sample_features).ravel()

# Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
sample_predictions_model5 = (sample_probabilities_model5 >= 0.5).astype(int)

print("--- Sample Features (First 5 from X_test_scaled) ---")
print(X_test.iloc[sample_indices]) # Display original features for context
print("\n--- Predicted Probabilities of Crime Occurrence (Model5) ---")
print(sample_probabilities_model5)
print("\n--- Predicted Crime Occurrence (0=No Crime, 1=Crime) (Model5) ---")
print(sample_predictions_model5)
print("\n--- Actual Crime Occurrence for These Samples ---")
print(y_test.iloc[sample_indices].values)

### Plotting Training History for Model 5

In [ ]:
import matplotlib.pyplot as plt

# Extract data from history5 object
acc5 = history5.history['accuracy']
val_acc5 = history5.history['val_accuracy']
loss5 = history5.history['loss']
val_loss5 = history5.history['val_loss']
epochs_range5 = range(len(acc5)) # Get number of epochs from history object

plt.figure(figsize=(12, 5))

# Plot Training and Validation Accuracy for model5
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_range5, acc5, label='Training Accuracy')
plt.plot(epochs_range5, val_acc5, label='Validation Accuracy')
plt.title('Model 5: Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Plot Training and Validation Loss for model5
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_range5, loss5, label='Training Loss')
plt.plot(epochs_range5, val_loss5, label='Validation Loss')
plt.title('Model 5: Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

print("Training and Validation Accuracy/Loss plots for Model 5 displayed.")

### Visualization of Model 5 Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Assuming y_test, y_pred5 (from model5 evaluation) are available
# If not, please run the evaluation cell for model5 again (cell 9cd3a48e)

cm5 = confusion_matrix(y_test, y_pred5)

plt.figure(figsize=(6, 5))
sns.heatmap(cm5, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0 (No Crime)', 'Predicted 1 (Crime)'],
            yticklabels=['Actual 0 (No Crime)', 'Actual 1 (Crime)'])
plt.title('Confusion Matrix for Model 5')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Confusion Matrix Heatmap for Model 5 displayed.")

### Model 6: Deeper Network with Batch Normalization, Dropout, SMOTE, and Early Stopping (Improved)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping


early_stop_model6 = EarlyStopping(
    monitor='val_loss',
    patience=7, # Increased patience for this model
    restore_best_weights=True
)

# Define Model 6 with an improved, deeper architecture, Batch Normalization, and Dropout
model6 = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model with Adam optimizer and binary_crossentropy loss
optimizer_model6 = Adam(learning_rate=0.0001)
model6.compile(optimizer=optimizer_model6, loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model6.summary()

# Train the model with SMOTE data and Early Stopping
history6 = model6.fit(
    X_train_resampled, y_train_resampled,
    epochs=50, # More epochs, relying on early stopping
    batch_size=128,
    verbose=1,
    callbacks=[early_stop_model6],
    validation_data=(X_test_scaled, y_test)
)

print("\nNeural network training complete for model6.")

### Predicting Crime Occurrences for the Full Test Set with Model 6

In [ ]:
# Predict probabilities on the full X_test_scaled using model6
y_prob_full_test_model6 = model6.predict(X_test_scaled).ravel()

# Convert probabilities to class labels (0 or 1) using a threshold of 0.5
y_pred_full_test_model6 = (y_prob_full_test_model6 >= 0.5).astype(int)

# Evaluation metrics
print("Classification Report for Model 6 on Full Test Set:")
print(classification_report(y_test, y_pred_full_test_model6, zero_division=0))
print("ROC-AUC for Model 6 on Full Test Set:", roc_auc_score(y_test, y_prob_full_test_model6))
print("Confusion Matrix for Model 6 on Full Test Set:\n", confusion_matrix(y_test, y_pred_full_test_model6))

Model 6 achieved higher o averall accuracy and a lower F1-Score for crime occurrence of 0.47 compared to Model 5

# **Visualization of Model 6 Confusion Matrix**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Assuming y_test and y_pred_full_test_model6 are available from the previous execution
cm6 = confusion_matrix(y_test, y_pred_full_test_model6)

plt.figure(figsize=(6, 5))
sns.heatmap(cm6, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0 (No Crime)', 'Predicted 1 (Crime)'],
            yticklabels=['Actual 0 (No Crime)', 'Actual 1 (Crime)'])
plt.title('Confusion Matrix for Model 6')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Confusion Matrix Heatmap for Model 6 displayed.")

# **Plotting Training History for Model 6**

In [ ]:
import matplotlib.pyplot as plt

# Extract data from history6 object
acc6 = history6.history['accuracy']
val_acc6 = history6.history['val_accuracy']
loss6 = history6.history['loss']
val_loss6 = history6.history['val_loss']
epochs_range6 = range(len(acc6)) # Get number of epochs from history object

plt.figure(figsize=(12, 5))

# Plot Training and Validation Accuracy for model6
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_range6, acc6, label='Training Accuracy')
plt.plot(epochs_range6, val_acc6, label='Validation Accuracy')
plt.title('Model 6: Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Plot Training and Validation Loss for model6
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_range6, loss6, label='Training Loss')
plt.plot(epochs_range6, val_loss6, label='Validation Loss')
plt.title('Model 6: Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

print("Training and Validation Accuracy/Loss plots for Model 6 displayed.")

# **Model 7: Deeper Network with Enhanced Regularization and Learning Rate Adjustment**

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Re-define EarlyStopping for clarity, potentially with adjusted patience
early_stop_model7 = EarlyStopping(
    monitor='val_loss',
    patience=8, # Slightly increased patience
    restore_best_weights=True
)

# Define Model 7 with a deeper architecture and enhanced regularization
model7 = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model with Adam optimizer and a slightly lower learning rate
optimizer_model7 = Adam(learning_rate=0.00005) # Further reduced learning rate
model7.compile(optimizer=optimizer_model7, loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model7.summary()

# Train the model with SMOTE data and Early Stopping
history7 = model7.fit(
    X_train_resampled, y_train_resampled,
    epochs=100, # More epochs, relying heavily on early stopping
    batch_size=128,
    verbose=1,
    callbacks=[early_stop_model7],
    validation_data=(X_test_scaled, y_test)
)

print("\nNeural network training complete for model7.")

### Evaluate Model 7 Performance

In [ ]:
# Predict probabilities for model7
y_prob7 = model7.predict(X_test_scaled).ravel()

# Convert probabilities to class labels for model7
y_pred7 = (y_prob7 >= 0.5).astype(int)

# Evaluation metrics for model7
print("Classification Report for Model 7:")
print(classification_report(y_test, y_pred7, zero_division=0))
print("ROC-AUC for Model 7:", roc_auc_score(y_test, y_prob7))
print("Confusion Matrix for Model 7:\n", confusion_matrix(y_test, y_pred7))

Moddel 7 has similar results to that of model 6

# **Training History**

In [ ]:
# Extract data from history7 object
acc7 = history7.history['accuracy']
val_acc7 = history7.history['val_accuracy']
loss7 = history7.history['loss']
val_loss7 = history7.history['val_loss']
epochs_range7 = range(len(acc7)) # Get number of epochs from history object

plt.figure(figsize=(12, 5))

# Plot Training and Validation Accuracy for model7
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_range7, acc7, label='Training Accuracy')
plt.plot(epochs_range7, val_acc7, label='Validation Accuracy')
plt.title('Model 7: Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Plot Training and Validation Loss for model7
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_range7, loss7, label='Training Loss')
plt.plot(epochs_range7, val_loss7, label='Validation Loss')
plt.title('Model 7: Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

print("Training and Validation Accuracy/Loss plots for Model 7 displayed.")

# **Visualization of Model 7 Confusion Matrix**

In [ ]:
cm7 = confusion_matrix(y_test, y_pred7)

plt.figure(figsize=(6, 5))
sns.heatmap(cm7, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0 (No Crime)', 'Predicted 1 (Crime)'],
            yticklabels=['Actual 0 (No Crime)', 'Actual 1 (Crime)'])
plt.title('Confusion Matrix for Model 7')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Confusion Matrix Heatmap for Model 7 displayed.")

# **Model 8 with SGD Optimizer**

In [ ]:
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping


early_stop_model8 = EarlyStopping(
    monitor='val_loss',
    patience=7, # Increased patience for this model
    restore_best_weights=True
)

# Define Model 8 with an improved, deeper architecture, Batch Normalization, and Dropout
model8 = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model with SGD optimizer and binary_crossentropy loss
optimizer_model8 = SGD(learning_rate=0.01, momentum=0.9)
model8.compile(optimizer=optimizer_model8, loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model8.summary()

# Train the model with SMOTE data and Early Stopping
history8 = model8.fit(
    X_train_resampled, y_train_resampled,
    epochs=50, # More epochs, relying on early stopping
    batch_size=128,
    verbose=1,
    callbacks=[early_stop_model8],
    validation_data=(X_test_scaled, y_test)
)

print("\nNeural network training complete for model8.")

# **Evaluation of Model 8 Performance**

In [ ]:
# Predict probabilities
y_prob8 = model8.predict(X_test_scaled).ravel()

# Convert probabilities to class labels
y_pred8 = (y_prob8 >= 0.5).astype(int)

# Evaluation metrics
print("Classification Report for Model 8:")
print(classification_report(y_test, y_pred8, zero_division=0))
print("ROC-AUC for Model 8:", roc_auc_score(y_test, y_prob8))
print("Confusion Matrix for Model 8:\n", confusion_matrix(y_test, y_pred8))

Model 8, using SGD, showed a slightly lower overall accuracy but managed to maintain a good F1-score for Class 1, achieving a high recall.
 Its F1-Score for crime occurrence was 0.48.

## **Model 9 with Deep Layers,Adam Optmizer and Removing SMOTE**

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Re-define EarlyStopping for clarity, potentially with addjusted patience
early_stop_model9 = EarlyStopping(
    monitor='val_loss',
    patience=8, # Slightly increased patience
    restore_best_weights=True
)

# Define Model 9 with a deeper architecture and enhanced regularization
model9 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model with Adam optimizer and a slightly lower learning rate
optimizer_model9 = Adam(learning_rate=0.00005)
model9.compile(optimizer=optimizer_model9, loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model9.summary()

# Train the model without SMOTE data and with Early Stopping
history9 = model9.fit(
    X_train_scaled, y_train,
    epochs=100, # More epochs, relying heavily on early stopping
    batch_size=128,
    verbose=1,
    callbacks=[early_stop_model9],
    validation_data=(X_test_scaled, y_test)
)

print("\nNeural network training complete for model9.")

# **Evaluation Of Model 9 Performance**

In [ ]:
# Predict probabilities for model9
y_prob9 = model9.predict(X_test_scaled).ravel()

# Convert probabilities to class labels for model9
y_pred9 = (y_prob9 >= 0.5).astype(int)

# Evaluation metrics for model9
print("Classification Report for Model 9:")
print(classification_report(y_test, y_pred9, zero_division=0))
print("ROC-AUC for Model 9:", roc_auc_score(y_test, y_prob9))
print("Confusion Matrix for Model 9:\n", confusion_matrix(y_test, y_pred9))

Model 9 stands out with the highest overall accuracy and very high precision for Class 0 (no crime), meaning it's excellent at correctly identifying no-crime instances and minimizing false alarms. However, this comes at a significant cost to recall for Class 1 (crime occurrence), as it misses a large portion of actual crime events, reflected in its lower F1-Score of 0.32.

# **Plotting Training History for Model 9**

In [ ]:
# Extract data from history9 object
acc9 = history9.history['accuracy']
val_acc9 = history9.history['val_accuracy']
loss9 = history9.history['loss']
val_loss9 = history9.history['val_loss']
epochs_range9 = range(len(acc9)) # Get number of epochs from history object

plt.figure(figsize=(12, 5))

# Plot Training and Validation Accuracy for model9
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_range9, acc9, label='Training Accuracy')
plt.plot(epochs_range9, val_acc9, label='Validation Accuracy')
plt.title('Model 9: Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Plot Training and Validation Loss for model9
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_range9, loss9, label='Training Loss')
plt.plot(epochs_range9, val_loss9, label='Validation Loss')
plt.title('Model 9: Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

print("Training and Validation Accuracy/Loss plots for Model 9 displayed.")

# **Visualization of Model 9 Confusion Matrix**

In [ ]:
cm9 = confusion_matrix(y_test, y_pred9)

plt.figure(figsize=(6, 5))
sns.heatmap(cm9, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0 (No Crime)', 'Predicted 1 (Crime)'],
            yticklabels=['Actual 0 (No Crime)', 'Actual 1 (Crime)'])
plt.title('Confusion Matrix for Model 9')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Confusion Matrix Heatmap for Model 9 displayed.")

# **Model 10 Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
model10 = LogisticRegression(
    solver='saga',
    random_state=42,
    class_weight='balanced', # To handle class imbalance
    C = 0.001, penalty='l2'
)

model10.fit(X_train_scaled, y_train)

print("Logistic Regression model training complete.")

# Predict probabilities on the scaled test set
y_prob_log_reg = model10.predict_proba(X_test_scaled)[:, 1]

# Convert probabilities to class labels (0 or 1) using a threshold of 0.5
y_pred_log_reg = (y_prob_log_reg >= 0.5).astype(int)

# Evaluation metrics
print("Classification Report for Logistic Regression Model:")
print(classification_report(y_test, y_pred_log_reg, zero_division=0))
print("ROC-AUC for Logistic Regression Model:", roc_auc_score(y_test, y_prob_log_reg))
print("Confusion Matrix for Logistic Regression Model:\n", confusion_matrix(y_test, y_pred_log_reg))

# **Feature Importance of Model 10**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances (coefficients) from the trained Logistic Regression model
coefficients = model10.coef_[0]

# Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': np.abs(coefficients) # Use absolute value for importance magnitude
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Plotting feature importances
plt.figure(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis', hue='Feature', legend=False)
plt.title('Feature Importance for Logistic Regression Model (L2)')
plt.xlabel('Absolute Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("Feature importance plot for the Logistic Regression model displayed.")

# **XG-BOOST**

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Best parameters found from RandomizedSearchCV (cell bBmHPY9YIjoM)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model (if not already done)
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = best_xgb_model.predict(X_test_scaled)

# Get metrics for the best XGBoost model
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.5)

# Get metrics for Model 8 and Model 9 as well
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.5)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.5)

# Combine all model metrics into a DataFrame
all_models_comparison_df = pd.DataFrame([metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9, metrics_xgb])
all_models_comparison_df = all_models_comparison_df.round(3)

print("\nOverall Model Performance Comparison (Threshold = 0.5):")
display(all_models_comparison_df)

All Model Performance Comparison at Threshold 0.5




In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = best_xgb_model.predict(X_test_scaled)

# --- Ensure fresh predictions for all models ---
# Predictions for Model 0, 1, 2, 3, 4, and 10 are already available as y_prob, y_prob1, etc. in previous cells.
# We will use these existing variables directly, assuming they hold the latest predictions.

y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.5)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.5)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.5)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.5)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.5)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.5)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.5)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.5)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.5)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.5)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.5)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.5)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nOverall Model Performance Comparison (Threshold = 0.5):\n(Refreshed output to ensure consistency)")
display(comparison_df)

All Model comparison at threshold of 0.6

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

def get_model_metrics(model_name, y_true, y_prob, threshold=0.6):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# --- Ensure fresh predictions for all models ---
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.6)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.6)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.6)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.6)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.6)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.6)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.6)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.6)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.6)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.6)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.6)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.6)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.6):\n(Refreshed output to ensure consistency)")
display(comparison_df)

Model Performance Comparison at Threshold 0.7

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.7):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# --- Ensure fresh predictions for all models ---
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.7)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.7)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.7)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.7)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.7)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.7)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.7)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.7)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.7)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.7)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.7)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.7)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.7):\n(Refreshed output to ensure consistency)")
display(comparison_df)

Model Performance Comparison at Threshold 0.8

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.8):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# --- Ensure fresh predictions for all models ---
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.8)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.8)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.8)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.8)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.8)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.8)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.8)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.8)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.8)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.8)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.8)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.8)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.8):\n(Refreshed output to ensure consistency)")
display(comparison_df)


Model Performance Comparison at Threshold 0.9

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.9):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# --- Ensure fresh predictions for all models ---
# The predictions for models 0, 1, 2, 3, 4 and 10 are already available as y_prob, y_prob1, etc. in previous cells.
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.9)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.9)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.9)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.9)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.9)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.9)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.9)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.9)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.9)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.9)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.9)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.9)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.9):\n(Refreshed output to ensure consistency)")
display(comparison_df)


Model Performance Comparison at Threshold 0.2

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.2):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# --- Ensure fresh predictions for all models ---
# The predictions for models 0, 1, 2, 3, 4 and 10 are already available as y_prob, y_prob1, etc. in previous cells.
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.2)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.2)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.2)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.2)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.2)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.2)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.2)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.2)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.2)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.2)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.2)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.2)

# Combine into a DataFrame
comparison_df = pd.DataFrame([metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4, metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9, metrics_model10, metrics_xgb])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.2):\n(Refreshed output to ensure consistency)")
display(comparison_df)

Model Performance Comparison at Threshold 0.3

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.3):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# --- Ensure fresh predictions for all models ---
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.3)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.3)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.3)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.3)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.3)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.3)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.3)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.3)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.3)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.3)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.3)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.3)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.3):\n")
display(comparison_df)

Model Performance Comparison at Threshold 0.4

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd
from xgboost import XGBClassifier

def get_model_metrics(model_name, y_true, y_prob, threshold=0.4):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

# Best parameters found from RandomizedSearchCV (re-defined here for self-containment)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_xgb_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_xgb_model.fit(X_train_scaled, y_train)

# Predict probabilities and labels for the best XGBoost model
y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# --- Ensure fresh predictions for all models ---
y_prob5_fresh = model5.predict(X_test_scaled).ravel()
y_prob6_fresh = model6.predict(X_test_scaled).ravel()
y_prob7_fresh = model7.predict(X_test_scaled).ravel()
y_prob8_fresh = model8.predict(X_test_scaled).ravel()
y_prob9_fresh = model9.predict(X_test_scaled).ravel()

# Get metrics for each neural network model using fresh predictions
metrics_model0 = get_model_metrics('Model 0', y_test, y_prob, threshold=0.4)
metrics_model1 = get_model_metrics('Model 1', y_test, y_prob1, threshold=0.4)
metrics_model2 = get_model_metrics('Model 2', y_test, y_prob2, threshold=0.4)
metrics_model3 = get_model_metrics('Model 3', y_test, y_prob3, threshold=0.4)
metrics_model4 = get_model_metrics('Model 4', y_test, y_prob4, threshold=0.4)
metrics_model5 = get_model_metrics('Model 5', y_test, y_prob5_fresh, threshold=0.4)
metrics_model6 = get_model_metrics('Model 6', y_test, y_prob6_fresh, threshold=0.4)
metrics_model7 = get_model_metrics('Model 7', y_test, y_prob7_fresh, threshold=0.4)
metrics_model8 = get_model_metrics('Model 8', y_test, y_prob8_fresh, threshold=0.4)
metrics_model9 = get_model_metrics('Model 9', y_test, y_prob9_fresh, threshold=0.4)
metrics_model10 = get_model_metrics('Model 10', y_test, y_prob_log_reg, threshold=0.4)
metrics_xgb = get_model_metrics('Best XGBoost', y_test, y_prob_xgb, threshold=0.4)

# Combine into a DataFrame
comparison_df = pd.DataFrame([
    metrics_model0, metrics_model1, metrics_model2, metrics_model3, metrics_model4,
    metrics_model5, metrics_model6, metrics_model7, metrics_model8, metrics_model9,
    metrics_model10, metrics_xgb
])
comparison_df = comparison_df.round(3)

print("\nNeural Network Model Performance Comparison (Threshold = 0.4):\n")
display(comparison_df)

Based on the performance comparison across various thresholds, a threshold of 0.3 generally provides a good balance between F1-Score for crime occurrence (Class 1) and overall accuracy across many of the models.

XGBoost, which was identified as the best performing model, achieves its highest F1-Score for Class 1 (0.525) at this threshold, with an accuracy of 0.707.
Several other Neural Network models (Model 0, 1, 2, 3, 7, 9) also achieve their peak F1-Scores around this 0.3 threshold (ranging from 0.501 to 0.509), with their accuracies generally falling in the 0.61-0.71 range, which can be considered good depending on the specific application's requirements.
While higher thresholds tend to increase precision for Class 1 (fewer false alarms) and overall accuracy for some models, they often lead to a significant drop in recall and thus a lower F1-Score for detecting actual crime events. Conversely, lower thresholds increase recall but drastically reduce precision and overall accuracy.

Therefore, 0.3 stands out as a sweet spot where models effectively balance identifying actual crime occurrences with maintaining reasonable overall accuracy.

In conclusion, the Best Neural Network Model at the 0.5 threshold is model 5.

### Best F1-Scores for Crime Occurrence (Class 1) and Corresponding Thresholds (Range 0.2-0.9)

In [ ]:
def get_best_f1_threshold_in_range(summary_df, model_name, min_threshold=0.2, max_threshold=0.9):
    filtered_df = summary_df[(summary_df['Threshold'] >= min_threshold) & (summary_df['Threshold'] <= max_threshold)]
    if filtered_df.empty:
        return {'Model': model_name, 'Best_Threshold': 'N/A', 'Max_F1-Score_1': 'N/A'}
    best_row = filtered_df.loc[filtered_df['F1-Score_1'].idxmax()]
    return {
        'Model': model_name,
        'Best_Threshold': best_row['Threshold'],
        'Max_F1-Score_1': best_row['F1-Score_1']
    }

all_models_best_f1_thresholds = []

# Neural Network Models
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model0, 'Model 0'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model1, 'Model 1'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model2, 'Model 2'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model3, 'Model 3'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model4, 'Model 4'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model5, 'Model 5'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model6, 'Model 6'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df, 'Model 7')) # summary_df holds Model 7 results
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model8, 'Model 8'))
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model9, 'Model 9'))

# Logistic Regression Model
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_model10, 'Model 10'))

# XGBoost Model
# For XGBoost, use its specific summary_df_xgb which might have more thresholds
all_models_best_f1_thresholds.append(get_best_f1_threshold_in_range(summary_df_xgb, 'XGBoost'))

# Create a DataFrame for display
comparison_best_f1_df = pd.DataFrame(all_models_best_f1_thresholds)
comparison_best_f1_df = comparison_best_f1_df.round(3)

print("\nBest F1-Scores for Crime Occurrence (Class 1) and Corresponding Thresholds (Range 0.2-0.9):")
display(comparison_best_f1_df)


The table above shows the best F1-Scores for Crime Occurrence (Class 1) and their corresponding thresholds for each model, ranging from 0.2 to 0.9.

 It appears that XGBoost achieved the highest F1-Score of 0.525 at a threshold of 0.25. Models 1, 8, 0, 5, 6, 7, and 9 also performed well, with F1-scores above 0.5 at various thresholds.

Model 10 (Logistic Regression) had a slightly lower best F1-score of 0.496. These results provide a comprehensive overview of how each model balances precision and recall for crime occurrence prediction across different decision thresholds.

The XGBoost model performed the best, achieving the highest F1-Score of 0.525 at a threshold of 0.3.

This indicates that XGBoost offered the most optimal balance between precision and recall in detecting crime occurrences among all the models evaluated. Additionally, it generally showed a strong ROC-AUC score, highlighting its overall discriminative ability

### Detailed F1-Score Comparison Across Specific Thresholds

In [ ]:
import pandas as pd

def evaluate_model_at_threshold_for_f1_comparison(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    f1_score_1 = report['1']['f1-score']
    return f1_score_1

# Define the specific thresholds to test
thresholds_to_test = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Dictionary to store F1-scores for each model at each threshold
model_f1_scores = {}

# List of (model_name, y_prob_variable)
models_to_compare = [
    ('Model 0', y_prob),
    ('Model 1', y_prob1),
    ('Model 2', y_prob2),
    ('Model 3', y_prob3),
    ('Model 4', y_prob4),
    ('Model 5', y_prob5_fresh), # Use fresh predictions
    ('Model 6', y_prob6_fresh), # Use fresh predictions
    ('Model 7', y_prob7_fresh), # Use fresh predictions
    ('Model 8', y_prob8_fresh), # Use fresh predictions
    ('Model 9', y_prob9_fresh), # Use fresh predictions
    ('Model 10', y_prob_log_reg),
    ('XGBoost', y_prob_xgb)
]

for model_name, y_prob_values in models_to_compare:
    f1_scores_for_model = []
    for threshold in thresholds_to_test:
        f1 = evaluate_model_at_threshold_for_f1_comparison(y_test, y_prob_values, threshold)
        f1_scores_for_model.append(f1)
    model_f1_scores[model_name] = f1_scores_for_model

# Create a DataFrame from the collected F1-scores
f1_comparison_df = pd.DataFrame(model_f1_scores, index=thresholds_to_test)
f1_comparison_df.index.name = 'Threshold'

print("\nF1-Score (Class 1) Comparison Across Specified Thresholds:")
display(f1_comparison_df.round(3))

Key Observations from the F1-Score Comparison Table:

At lower thresholds (0.2, 0.3):

XGBoost generally achieves the highest F1-scores for Class 1 (crime occurrence), reaching 0.513 at 0.2 and 0.525 at 0.3.
Model 1 and Model 9 also show strong F1-scores at these lower thresholds, both achieving 0.501 at 0.2 and 0.509 (Model 1) / 0.505 (Model 9) at 0.3.
Model 0, Model 2, Model 3, Model 5, Model 6, Model 7 and Model 8 show competitive F1-scores, ranging from 0.427 to 0.506 at thresholds 0.2 and 0.3.
Model 10 (Logistic Regression) also performs reasonably well at these lower thresholds, with F1-scores of 0.410 at 0.2 and 0.448 at 0.3.
At threshold 0.4:

Model 8 now leads with an F1-score of 0.506, closely followed by Model 5, Model 6 and Model 7 with F1-scores of 0.496, 0.502 and 0.502, respectively.
XGBoost's F1-score drops to 0.486, while Model 1 drops to 0.457 and Model 9 drops to 0.441.
At threshold 0.5:

Model 4, 5 and Model 8 show the highest F1-scores among all models, with 0.507 and 0.478, respectively. This suggests a good balance of precision and recall for crime occurrence at this threshold.
XGBoost and Model 9 experience significant drops in F1-score at this threshold (0.386 and 0.322 respectively), indicating that they prioritize precision for the 'no crime' class more strongly at higher thresholds.
At higher thresholds (0.6, 0.7, 0.8, 0.9):

F1-scores for most models generally decrease as the threshold increases, reflecting the trade-off between precision and recall. Higher thresholds lead to fewer false positives (higher precision) but also more false negatives (lower recall).
Conclusion:



### Comparison of Confusion Matrices for Models 5, 6, 7, 8 and 9

In [ ]:
print("Confusion Matrix for Model 5:\n", cm5)
print("\nConfusion Matrix for Model 6:\n", cm6)
print("\nConfusion Matrix for Model 7:\n", cm7)
cm8 = confusion_matrix(y_test, y_pred8) # Calculate cm8
print("\nConfusion Matrix for Model 8:\n", cm8)
print("\nConfusion Matrix for Model 9:\n", cm9)

### Heatmap Visualizations of Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure with subplots for each confusion matrix
# Changed from 1 row, 3 columns to 2 rows, 3 columns to accommodate 5 plots
fig, axes = plt.subplots(2, 3, figsize=(20, 12)) # Adjusted figsize for better viewing

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Confusion Matrix for Model 5
sns.heatmap(cm5, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'], ax=axes[0])
axes[0].set_title('Model 5 Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Confusion Matrix for Model 6
sns.heatmap(cm6, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'], ax=axes[1])
axes[1].set_title('Model 6 Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

# Confusion Matrix for Model 7
sns.heatmap(cm7, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'], ax=axes[2])
axes[2].set_title('Model 7 Confusion Matrix')
axes[2].set_xlabel('Predicted Label')
axes[2].set_ylabel('True Label')

# Confusion Matrix for Model 8
sns.heatmap(cm8, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'], ax=axes[3])
axes[3].set_title('Model 8 Confusion Matrix')
axes[3].set_xlabel('Predicted Label')
axes[3].set_ylabel('True Label')

# Confusion Matrix for Model 9
sns.heatmap(cm9, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'], ax=axes[4])
axes[4].set_title('Model 9 Confusion Matrix')
axes[4].set_xlabel('Predicted Label')
axes[4].set_ylabel('True Label')

# Hide the empty subplot if there is one
if len(axes) > 5:
    for i in range(5, len(axes)):
        fig.delaxes(axes[i])


plt.tight_layout()
plt.show()

print("Heatmap visualizations of Confusion Matrices for Models 5, 6, 7, 8 and 9 displayed.")

### Combined Performance Comparison at Different Thresholds

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

def evaluate_model_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    accuracy = report['accuracy']

    # Extract metrics for class 1 (crime occurrence)
    precision_1 = report['1']['precision']
    recall_1 = report['1']['recall']
    f1_score_1 = report['1']['f1-score']

    return {
        'Threshold': threshold,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'Precision_1': precision_1,
        'Recall_1': recall_1,
        'F1-Score_1': f1_score_1
    }

thresholds_to_test = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Recalculate summary_df for Model 7 (named 'summary_df' in the notebook context)
results_model7_for_plot = []
for threshold in thresholds_to_test:
    results_model7_for_plot.append(evaluate_model_at_threshold(y_test, y_prob7, threshold))
summary_df = pd.DataFrame(results_model7_for_plot)
summary_df = summary_df.round(3)

# Recalculate summary_df_model5
results_model5_for_plot = []
for threshold in thresholds_to_test:
    results_model5_for_plot.append(evaluate_model_at_threshold(y_test, y_prob5, threshold))
summary_df_model5 = pd.DataFrame(results_model5_for_plot)
summary_df_model5 = summary_df_model5.round(3)

# Recalculate summary_df_model6
results_model6_for_plot = []
for threshold in thresholds_to_test:
    results_model6_for_plot.append(evaluate_model_at_threshold(y_test, y_prob_full_test_model6, threshold))
summary_df_model6 = pd.DataFrame(results_model6_for_plot)
summary_df_model6 = summary_df_model6.round(3)

# Calculate summary_df_model8
results_model8_for_plot = []
for threshold in thresholds_to_test:
    results_model8_for_plot.append(evaluate_model_at_threshold(y_test, y_prob8, threshold))
summary_df_model8 = pd.DataFrame(results_model8_for_plot)
summary_df_model8 = summary_df_model8.round(3)

# Calculate summary_df_model9
results_model9_for_plot = []
for threshold in thresholds_to_test:
    results_model9_for_plot.append(evaluate_model_at_threshold(y_test, y_prob9, threshold))
summary_df_model9 = pd.DataFrame(results_model9_for_plot)
summary_df_model9 = summary_df_model9.round(3)

print("\nModel 5 Performance Summary (re-displayed for comparison):")
display(summary_df_model5)

print("\nModel 6 Performance Summary:")
display(summary_df_model6)

print("\nModel 7 Performance Summary:")
display(summary_df)

print("\nModel 8 Performance Summary:")
display(summary_df_model8)

print("\nModel 9 Performance Summary:")
display(summary_df_model9)

print("\nAnalysis of Model Performance Across Thresholds:\n")
print("To determine the 'best' model, we need to consider the specific objective (e.g., maximizing recall, precision, or F1-score for crime occurrence) and the acceptable trade-offs.\n")

print("**Key Observations:**\n")

print("**F1-Score for Class 1 (Crime Occurrence):**")
print("- **Model 5** generally achieves the highest F1-Score (e.g., 0.491 at threshold 0.45, 0.490 at 0.5) among the three models at higher thresholds, indicating a good balance between precision and recall for crime prediction.\n")
print("- **Model 6** shows a peak F1-Score of 0.473 at a threshold of 0.5, which is lower than Model 5, but it maintains competitive scores at lower thresholds.\n")
print("- **Model 7**'s F1-Score for Class 1 is comparable to Model 5 at some thresholds (e.g., 0.488 at 0.5), suggesting similar overall effectiveness in identifying crime occurrences.\n")
print("- **Model 8** also shows strong F1-Scores, reaching 0.493 at threshold 0.5, indicating a good balance especially when optimized with SGD.\n")
print("- **Model 9** exhibits lower F1-Scores at higher thresholds (e.g., 0.295 at 0.5), but shows a higher F1-score at lower thresholds, indicating a preference for identifying true negatives over true positives.\n")

print("**Recall for Class 1 (Identifying Actual Crimes):**")
print("- If the priority is to identify as many actual crime occurrences as possible (high recall), **Models 5, 7, and 8** tend to perform better at lower thresholds. For instance, at a threshold of 0.05, all models achieve very high recall (close to 1.0), but their precision is very low.\n")
print("- As the threshold increases, recall generally decreases, but Models 5, 7, and 8 maintain relatively higher recall compared to Model 6 and 9 for similar precision levels.\n")

print("**Precision for Class 1 (Minimizing False Alarms):**")
print("- If minimizing false alarms (high precision) is critical, all models show increasing precision as the threshold increases. **Model 9** stands out for its very high precision at higher thresholds (e.g., 0.613 at 0.5), indicating fewer false positives, but at the expense of recall.\n")
print("- **Model 6** often exhibits slightly higher precision at higher thresholds compared to Model 5, 7, and 8, but often at the expense of recall.\n")

print("**ROC-AUC:**")
print("- The ROC-AUC scores are quite similar across all models (around 0.718-0.730), indicating that their overall ability to distinguish between classes is comparable, regardless of the threshold.\n")

print("**Conclusion:**\n")
print("The 'best' model depends on the specific goals. If the primary goal is to **balance precision and recall** for crime occurrences, **Model 5, Model 7, or Model 8** appear to be slightly better due to their higher F1-scores around the 0.45-0.5 threshold. If **minimizing false positives is more important** (high precision), **Model 9** is preferred, but with an understanding of the trade-off in significantly lower recall. If the goal is simply to **catch as many crimes as possible** (high recall), a very low threshold on any of these models would achieve that, but with very low precision (many false alarms).")

### Summary of Model Performance Across Different Thresholds



**Key Observations:**

**Model 5 (Neural Network with SMOTE):**
- **F1-Score for Class 1:** Generally achieves competitive F1-Scores of 0.51 at 0.5 threshold, indicating a good balance between precision and recall for crime prediction.
- **Recall for Class 1:** Maintains relatively higher recall compared to Model 9 for similar precision levels, especially at lower thresholds.
- **Precision for Class 1:** Shows increasing precision as the threshold increases, with moderate false positives.

**Model 6 (Deeper Neural Network with Batch Normalization, Dropout, SMOTE):**
- **F1-Score for Class 1:** Shows Lower thresholds at 0.3, 0.4 achieve a high f1 sore of 0.5 which is slightly lower than Model 5, but it maintains competitive scores at lower thresholds.
- **Recall for Class 1:** Comparable to Model 5 at lower thresholds, but tends to decrease more sharply at higher thresholds.
- **Precision for Class 1:** Often exhibits slightly higher precision at higher thresholds compared to Model 5, 7, and 8, but often at the expense of recall.

**Model 7 (Deeper Neural Network with Enhanced Regularization, SMOTE):**
- **F1-Score for Class 1:** Comparable to Model 6 at some thresholds (e.g., 0.488 at 0.5), suggesting similar overall effectiveness in identifying crime occurrences. Its peak F1-score for crime was 0.50 0.50  at a 0.3 and 0.4 threshold.
- **Recall for Class 1:** Similar trends to Model 6, with higher recall at lower thresholds.
- **Precision for Class 1:** Increases with threshold, balancing false positives and true positives.

**Model 8 (Deeper Neural Network with SGD, SMOTE):**
- **F1-Score for Class 1:** Shows strong F1-Scores at low thresholds indicating a good balance, especially when optimized with SGD. This model achieved the highest recall among the SMOTE-enhanced models.
- **Recall for Class 1:** Highest recall among the SMOTE-enhanced models, making it good at identifying actual crime events.
- **Precision for Class 1:** Balances precision with its high recall, though it may have slightly more false positives than other SMOTE models at specific thresholds.

**Model 9 (Deeper Neural Network without SMOTE):**
- **F1-Score for Class 1:** Exhibits significantly lower F1-Scores at higher thresholds compared to SMOTE models. This indicates it misses a significant number of actual crime occurrences.
- **Recall for Class 1:** Very low recall at higher thresholds, meaning it identifies only a small fraction of actual crime events.
- **Precision for Class 1:** Stands out for its very high precision at higher thresholds, indicating fewer false positives for crime predictions and excellent performance in predicting 'no crime' instances.

**ROC-AUC across all models:**
- The ROC-AUC scores are quite similar across all models  indicating that their overall ability to distinguish between classes is comparable, regardless of the threshold.

### F1-Score of Models 5, 6, 7, 8 and 9 Across Different Thresholds

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.lineplot(x='Threshold', y='F1-Score_1', data=summary_df_model5, label='Model 5 F1-Score')
sns.lineplot(x='Threshold', y='F1-Score_1', data=summary_df_model6, label='Model 6 F1-Score')
sns.lineplot(x='Threshold', y='F1-Score_1', data=summary_df, label='Model 7 F1-Score')
sns.lineplot(x='Threshold', y='F1-Score_1', data=summary_df_model8, label='Model 8 F1-Score')
sns.lineplot(x='Threshold', y='F1-Score_1', data=summary_df_model9, label='Model 9 F1-Score')

plt.title('F1-Score for Crime Occurrence Across Different Thresholds')
plt.xlabel('Decision Threshold')
plt.ylabel('F1-Score (Class 1 - Crime Occurrence)')
plt.grid(True)
plt.legend()
plt.show()

print("F1-Score plot for Models 5, 6, 7, 8 and 9 displayed.")

Models that used smote have higher f1 scores with the least score being model 9 with no smote dataset.

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 12)) # Adjust figure size to accommodate five subplots

# Model 5 ROC Curve
plt.subplot(2, 3, 1) # 2 rows, 3 columns, first plot
fpr5, tpr5, _ = roc_curve(y_test, y_prob5_fresh)
roc_auc5 = auc(fpr5, tpr5)
plt.plot(fpr5, tpr5, color='darkorange', lw=2, label=f'Model 5 (AUC = {roc_auc5:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Model 5')
plt.legend(loc='lower right')
plt.grid(True)

# Model 6 ROC Curve
plt.subplot(2, 3, 2) # 2 rows, 3 columns, second plot
fpr6, tpr6, _ = roc_curve(y_test, y_prob6_fresh)
roc_auc6 = auc(fpr6, tpr6)
plt.plot(fpr6, tpr6, color='green', lw=2, label=f'Model 6 (AUC = {roc_auc6:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Model 6')
plt.legend(loc='lower right')
plt.grid(True)

# Model 7 ROC Curve
plt.subplot(2, 3, 3) # 2 rows, 3 columns, third plot
fpr7, tpr7, _ = roc_curve(y_test, y_prob7_fresh)
roc_auc7 = auc(fpr7, tpr7)
plt.plot(fpr7, tpr7, color='blue', lw=2, label=f'Model 7 (AUC = {roc_auc7:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Model 7')
plt.legend(loc='lower right')
plt.grid(True)

# Model 8 ROC Curve
plt.subplot(2, 3, 4) # 2 rows, 3 columns, fourth plot
fpr8, tpr8, _ = roc_curve(y_test, y_prob8_fresh)
roc_auc8 = auc(fpr8, tpr8)
plt.plot(fpr8, tpr8, color='purple', lw=2, label=f'Model 8 (AUC = {roc_auc8:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Model 8')
plt.legend(loc='lower right')
plt.grid(True)

# Model 9 ROC Curve
plt.subplot(2, 3, 5) # 2 rows, 3 columns, fifth plot
fpr9, tpr9, _ = roc_curve(y_test, y_prob9_fresh)
roc_auc9 = auc(fpr9, tpr9)
plt.plot(fpr9, tpr9, color='red', lw=2, label=f'Model 9 (AUC = {roc_auc9:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Model 9')
plt.legend(loc='lower right')
plt.grid(True)

plt.tight_layout() # Adjust layout to prevent overlap
plt.show()

print("Separate ROC Curves for Models 5, 6, 7, 8, and 9 displayed.")

### Summary of Crime Occurrence Predictions for Models 5, 6, 7, 8 and 9 on Sample Data

In [ ]:
# Select a few samples from the scaled test set for demonstration (same as used for Model 5 individual demo)
sample_indices = list(range(15)) # Using the first 15 samples from X_test_scaled
sample_features = X_test_scaled[sample_indices]

# Make predictions (probabilities) using each model
sample_probabilities_model5 = model5.predict(sample_features).ravel()
sample_probabilities_model6 = model6.predict(sample_features).ravel()
sample_probabilities_model7 = model7.predict(sample_features).ravel()
sample_probabilities_model8 = model8.predict(sample_features).ravel()
sample_probabilities_model9 = model9.predict(sample_features).ravel()

# Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
sample_predictions_model5 = (sample_probabilities_model5 >= 0.5).astype(int)
sample_predictions_model6 = (sample_probabilities_model6 >= 0.5).astype(int)
sample_predictions_model7 = (sample_probabilities_model7 >= 0.5).astype(int)
sample_predictions_model8 = (sample_probabilities_model8 >= 0.5).astype(int)
sample_predictions_model9 = (sample_probabilities_model9 >= 0.5).astype(int)

# Get the actual crime occurrences for these samples
actual_occurrence = y_test.iloc[sample_indices].values

# Create a DataFrame to display the comparison
prediction_summary_df = pd.DataFrame({
    'Community Area': X_test.iloc[sample_indices]['Community Area'].values,
    'Actual Occurrence': actual_occurrence,
    'Model 5 Prob': sample_probabilities_model5,
    'Model 5 Pred': sample_predictions_model5,
    'Model 6 Prob': sample_probabilities_model6,
    'Model 6 Pred': sample_predictions_model6,
    'Model 7 Prob': sample_probabilities_model7,
    'Model 7 Pred': sample_predictions_model7,
    'Model 8 Prob': sample_probabilities_model8,
    'Model 8 Pred': sample_predictions_model8,
    'Model 9 Prob': sample_probabilities_model9,
    'Model 9 Pred': sample_predictions_model9
})

print("Comparison of Crime Occurrence Predictions for Sample Data:")
display(prediction_summary_df)


In [ ]:
pip install shap

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Assuming 'feature_cols' from earlier cells defines the feature names
# Convert X_test_scaled back to a DataFrame for better SHAP plotting
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols)

# Create a smaller background dataset for DeepExplainer for efficiency
# Use a random subset of X_train_resampled, as model5 was trained on it
background_data_size = 1000 # Using 1000 samples for background, can be adjusted
# Ensure random state for reproducibility if needed, though DeepExplainer doesn't have it directly for sampling
background_indices = np.random.choice(X_train_resampled.shape[0], background_data_size, replace=False)
background_data = X_train_resampled[background_indices]

# Create a SHAP explainer for Model 5 (Keras model)
# DeepExplainer is suitable for deep learning models
explainer = shap.DeepExplainer(model5, background_data)

# Calculate SHAP values for the test set
# This can be computationally intensive, so we'll use a subset for demonstration
sample_indices_shap = list(range(100)) # Use first 100 samples for reasonable computation time
shap_values = explainer.shap_values(X_test_scaled[sample_indices_shap])

print("SHAP values calculated.")


### SHAP Summary Plot for Model 5

The SHAP summary plot provides a comprehensive overview of feature importance and impact. Each point on the plot is a Shapley value for a feature and an instance. The position on the x-axis shows the impact of that feature on the model's output prediction. The color represents the feature's value (red for high, blue for low).

In [ ]:
# Visualize the SHAP values
# shap_values from DeepExplainer with a Keras model often comes with an extra dimension for output classes.
# For binary classification with a single sigmoid output, it might be (num_samples, num_features, 1).
# We need to squeeze this to (num_samples, num_features).
shap_values_squeezed = np.array(shap_values).squeeze()

# Ensure the squeezed SHAP values are 2D if they ended up 1D (e.g., if only one sample was explained)
if shap_values_squeezed.ndim == 1:
    shap_values_squeezed = shap_values_squeezed.reshape(1, -1)

shap.summary_plot(
    shap_values_squeezed,
    X_test_scaled_df.iloc[sample_indices_shap],
    plot_type="dot",
    show=False,
    feature_names=feature_cols # Add feature names for better readability
)
plt.title('SHAP Summary Plot for Model 5 (Crime Occurrence)')
plt.tight_layout()
plt.show()

print("SHAP summary plot for Model 5 displayed.")

Based on the SHAP summary plot for Model 5, the following observations can be made about feature importance and impact on crime occurrence predictions:

Lag Features (e.g., lag_1h, lag_2h, lag_3h, lag_24h) and Rolling Mean Features (rolling_3h_mean, rolling_6h_mean, rolling_12h_mean, rolling_24h_mean): These are consistently the most important features. High values (red dots) for these features, indicating recent or historical crime activity, tend to push the model's prediction towards a higher probability of crime occurrence. Conversely, low values (blue dots) of these features reduce the predicted crime probability.

Cyclical Hour Features (hour_sin, hour_cos): These features representing the hour of the day are also highly influential. Their impact can vary, with certain hours (indicated by specific hour_sin and hour_cos values) either increasing (red dots to the right) or decreasing (blue dots to the left) the likelihood of a crime.

Community Area: The geographical location is a significant predictor. Specific community areas (represented by their values) have a strong influence on the crime prediction, suggesting that some areas are inherently more prone to crime.

Day of Week, Month, Day, and Is Weekend: These temporal features also contribute to the prediction but generally show a lesser magnitude of influence compared to the lag, rolling mean, and cyclical hour features. For example, certain days of the week or months might slightly increase or decrease the crime probability.

In essence, Model 5 heavily relies on past crime patterns (lags and rolling means), the time of day, and the specific community area to predict crime occurrences. Features related to specific days or months also play a role, but with less pronounced impact.

### SHAP Analysis for Model 6 (Neural Network)

To ensure consistent analysis, we will use the probabilities from the fully trained `model6` (i.e., `y_prob_full_test_model6`) for SHAP interpretation.


In [ ]:
# Create a SHAP explainer for Model 6 (Keras model)
explainer6 = shap.DeepExplainer(model6, background_data)

# Calculate SHAP values for the test set subset
shap_values6 = explainer6.shap_values(X_test_scaled[sample_indices_shap])

print("SHAP values calculated for Model 6.")


### SHAP Summary Plot for Model 6
The SHAP summary plot for Model 6 provides insights into feature importance and direction of impact on its predictions.

In [ ]:
# Visualize the SHAP values for Model 6
shap_values_squeezed6 = np.array(shap_values6).squeeze()

if shap_values_squeezed6.ndim == 1:
    shap_values_squeezed6 = shap_values_squeezed6.reshape(1, -1)

shap.summary_plot(
    shap_values_squeezed6,
    X_test_scaled_df.iloc[sample_indices_shap],
    plot_type="dot",
    show=False,
    feature_names=feature_cols
)
plt.title('SHAP Summary Plot for Model 6 (Crime Occurrence)')
plt.tight_layout()
plt.show()

print("SHAP summary plot for Model 6 displayed.")

Based on the SHAP summary plot for Model 6, the following key observations can be made:

*   **Lag Features (`lag_1h`, `lag_2h`, `lag_3h`, `lag_24h`) and Rolling Mean Features (`rolling_3h_mean`, `rolling_6h_mean`, `rolling_12h_mean`, `rolling_24h_mean`):** These features are highly prominent and consistently rank as the most influential. High values (red dots) for these features, representing historical crime activity, strongly contribute to a higher predicted probability of crime. Conversely, low values (blue dots) reduce the predicted crime likelihood.

*   **Cyclical Hour Features (`hour_sin`, `hour_cos`):** The features encoding the hour of the day show significant influence. Specific `hour_sin` and `hour_cos` values can either increase or decrease the predicted crime occurrence, reflecting varying crime rates throughout the day.

*   **Community Area:** The geographical location remains a strong predictor, indicating that certain community areas are more prone to crime occurrences, influencing predictions irrespective of other factors.

*   **Day of Week, Month, Day, and Is Weekend:** These temporal features also play a role, but their magnitude of influence is generally less pronounced compared to the lag, rolling mean, and cyclical hour features. They help capture weekly, monthly, and yearly crime trends.

In summary, Model 6, like Model 5, relies heavily on past crime patterns, the specific time of day, and the community area for its predictions. The deeper architecture and regularization in Model 6 aim to refine these insights.

### SHAP Analysis for Model 7 (Neural Network)

In [ ]:
# Create a SHAP explainer for Model 7 (Keras model)
explainer7 = shap.DeepExplainer(model7, background_data)

# Calculate SHAP values for the test set subset
shap_values7 = explainer7.shap_values(X_test_scaled[sample_indices_shap])

print("SHAP values calculated for Model 7.")

### SHAP Summary Plot for Model 7
The SHAP summary plot for Model 7 shows its feature importance and how features influence its predictions.

In [ ]:
# Visualize the SHAP values for Model 7
shap_values_squeezed7 = np.array(shap_values7).squeeze()

if shap_values_squeezed7.ndim == 1:
    shap_values_squeezed7 = shap_values_squeezed7.reshape(1, -1)

shap.summary_plot(
    shap_values_squeezed7,
    X_test_scaled_df.iloc[sample_indices_shap],
    plot_type="dot",
    show=False,
    feature_names=feature_cols
)
plt.title('SHAP Summary Plot for Model 7 (Crime Occurrence)')
plt.tight_layout()
plt.show()

print("SHAP summary plot for Model 7 displayed.")

For Model 7, the SHAP summary plot reveals a similar pattern of feature importance:

*   **Lag Features (`lag_1h`, `lag_2h`, `lag_3h`, `lag_24h`) and Rolling Mean Features (`rolling_3h_mean`, `rolling_6h_mean`, `rolling_12h_mean`, `rolling_24h_mean`):** These features continue to be the most critical drivers of the model's predictions. High values consistently push the output towards a higher probability of crime, signifying the strong predictive power of recent and historical crime events.

*   **Cyclical Hour Features (`hour_sin`, `hour_cos`):** These features are highly influential, capturing the hourly variations in crime likelihood. Different times of the day (represented by these values) have a clear impact on whether crime is predicted.

*   **Community Area:** This feature maintains its importance, demonstrating that the geographical context of a community area is a fundamental factor in predicting crime occurrences.

*   **Day of Week, Month, Day, and Is Weekend:** While contributing, these features have a relatively smaller impact compared to the primary drivers. They still offer insights into weekly, monthly, and seasonal crime patterns.

Model 7, with its enhanced regularization and deeper network, confirms that historical crime data, time of day, and location are the dominant predictive factors, striving for a more robust and generalized understanding of these relationships.

### SHAP Analysis for Model 8 (Neural Network)


In [ ]:
# Create a SHAP explainer for Model 8 (Keras model)
explainer8 = shap.DeepExplainer(model8, background_data)

# Calculate SHAP values for the test set subset
shap_values8 = explainer8.shap_values(X_test_scaled[sample_indices_shap])

print("SHAP values calculated for Model 8.")


### SHAP Summary Plot for Model 8
The SHAP summary plot for Model 8 shows its feature importance and how features influence its predictions.

In [ ]:
# Visualize the SHAP values for Model 8
shap_values_squeezed8 = np.array(shap_values8).squeeze()

if shap_values_squeezed8.ndim == 1:
    shap_values_squeezed8 = shap_values_squeezed8.reshape(1, -1)

shap.summary_plot(
    shap_values_squeezed8,
    X_test_scaled_df.iloc[sample_indices_shap],
    plot_type="dot",
    show=False,
    feature_names=feature_cols
)
plt.title('SHAP Summary Plot for Model 8 (Crime Occurrence)')
plt.tight_layout()
plt.show()

print("SHAP summary plot for Model 8 displayed.")


The SHAP summary plot for Model 8, which utilized the SGD optimizer, presents a consistent picture of feature importance:

*   **Lag Features (`lag_1h`, `lag_2h`, `lag_3h`, `lag_24h`) and Rolling Mean Features (`rolling_3h_mean`, `rolling_6h_mean`, `rolling_12h_mean`, `rolling_24h_mean`):** These features once again stand out as the most impactful. High values of these indicators consistently lead to increased predictions of crime, underlining the significance of past crime occurrences.

*   **Cyclical Hour Features (`hour_sin`, `hour_cos`):** The representation of the hour of the day through `hour_sin` and `hour_cos` remains highly influential, capturing the critical daily periodicity of crime events.

*   **Community Area:** The `Community Area` feature continues to be a crucial determinant, reflecting that crime prediction is fundamentally linked to specific geographical locations.

*   **Day of Week, Month, Day, and Is Weekend:** These broader temporal features are relevant but generally less impactful than the highly specific lag and rolling mean features. They provide context regarding weekly and monthly crime trends.

Model 8, despite using SGD, confirms that the underlying predictive structure is heavily driven by historical crime, time of day, and location, reinforcing the findings from previous models.

Interpreting the Plot:


 The SHAP analysis confirms that these spatial and temporal features are highly influential in the models' predictions.
 By looking at the plot, you can identify which features have the strongest influence on whether the model predicts a crime occurrence. For example:

If you see many red dots (high feature values) extending to the right for a particular feature, it means high values of that feature tend to increase the probability of crime occurrence.
If you see many blue dots (low feature values) extending to the left, it means low values of that feature tend to decrease the probability of crime occurrence.

### SHAP Analysis for Model 9 (Neural Network)

In [ ]:
# Create a SHAP explainer for Model 9 (Keras model)
explainer9 = shap.DeepExplainer(model9, background_data)

# Calculate SHAP values for the test set subset
shap_values9 = explainer9.shap_values(X_test_scaled[sample_indices_shap])

print("SHAP values calculated for Model 9.")

### SHAP Summary Plot for Model 9
The SHAP summary plot for Model 9 shows its feature importance and how features influence its predictions.

In [ ]:
# Visualize the SHAP values for Model 9
shap_values_squeezed9 = np.array(shap_values9).squeeze()

if shap_values_squeezed9.ndim == 1:
    shap_values_squeezed9 = shap_values_squeezed9.reshape(1, -1)

shap.summary_plot(
    shap_values_squeezed9,
    X_test_scaled_df.iloc[sample_indices_shap],
    plot_type="dot",
    show=False,
    feature_names=feature_cols
)
plt.title('SHAP Summary Plot for Model 9 (Crime Occurrence)')
plt.tight_layout()
plt.show()

print("SHAP summary plot for Model 9 displayed.")

The SHAP summary plot for Model 9, which was trained *without* SMOTE oversampling, shows similar yet distinct observations regarding feature importance:

*   **Lag Features (`lag_1h`, `lag_2h`, `lag_3h`, `lag_24h`) and Rolling Mean Features (`rolling_3h_mean`, `rolling_6h_mean`, `rolling_12h_mean`, `rolling_24h_mean`):** These features are still the most significant predictors. High values of these historical crime indicators push the model's prediction towards a higher probability of crime. However, the model, without SMOTE, might be more sensitive to patterns in the majority 'no crime' class.

*   **Cyclical Hour Features (`hour_sin`, `hour_cos`):** The hourly cyclical features maintain their high importance, illustrating the strong daily patterns in crime occurrence that the model captures.

*   **Community Area:** The geographical context of the `Community Area` remains a fundamental predictor, signifying the inherent crime levels associated with different locations.

*   **Day of Week, Month, Day, and Is Weekend:** These temporal features contribute to the predictions, but their influence is comparatively lower than the more direct historical and hourly indicators.

While Model 9 identifies the same top features as the SMOTE-enhanced models, its training on an imbalanced dataset often leads to a model that excels at identifying the majority class (no crime) at the expense of recall for the minority class (crime). The SHAP values reflect how these features influence this specific behavior, often showing stronger impacts when features align with the majority class patterns or very strong signals for the minority class.

Key Observations Across Models:

Time-Based Features (rolling_mean, lag, hour_sin, hour_cos): These features consistently appear as the most influential across all models (Neural Networks and XGBoost). This strongly suggests that past crime occurrences and cyclical time patterns (like hour of the day) are critical predictors for future crime events.


Rolling Means and Lag Features: High values of rolling_24h_mean, rolling_6h_mean, rolling_12h_mean, and lag_24h often push the prediction towards a higher probability of crime, indicating that recent and historical crime activity in a community area is a strong indicator.


Cyclical Hour Features (hour_sin, hour_cos): These are important in capturing daily patterns. Depending on the hour, these features can either increase or decrease the predicted crime occurrence, reflecting that certain hours of the day are more prone to crime than others.


Geographical Feature (Community Area): The Community Area feature is also consistently important across all models. This indicates that crime prediction is highly location-dependent, and certain community areas are inherently more susceptible to crime, irrespective of other factors.

Day/Month/Weekend Features: day_of_week, is_weekend, month, and day also contribute to predictions, but generally to a lesser extent than the rolling/lag and cyclical hour features. They capture weekly, monthly, and yearly trends in crime.

Model-Specific Nuances:
Neural Network Models (Model 5, 6, 7): While the top features remain consistent, the specific magnitudes and exact ordering of influence can vary slightly between the neural network models. For instance, some models might weigh rolling_24h_mean slightly higher than rolling_6h_mean or vice-versa, but the overall picture of time-series dependencies dominating remains true.


In conclusion, the SHAP analysis provides strong evidence that a combination of historical crime data (lag and rolling features), temporal patterns (hour, day of week, month), and geographical context (community area) are the most critical factors driving the models' predictions for crime occurrence. This interpretability helps validate the models' decisions and provides insights into the underlying dynamics of crime prediction.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

# Best parameters found from RandomizedSearchCV (cell bBmHPY9YIjoM)
best_params = {
    'subsample': 0.9,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'n_estimators': 800,
    'max_depth': 4,
    'learning_rate': 0.1,
    'gamma': 3,
    'colsample_bytree': 0.7
}

# Re-instantiate and fit the best XGBoost model
best_model = XGBClassifier(
    random_state=42, # Ensure reproducibility
    tree_method='hist', # Use the same tree method as before
    **best_params
)

best_model.fit(X_train_scaled, y_train)

# Get feature importances from the best XGBoost model
feature_importances = best_model.feature_importances_

# Get feature names from X (or X_train)
feature_names = X.columns.tolist()

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# Sort by importance
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plotting feature importances
plt.figure(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Feature Importance for Best XGBoost Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("Feature importance plot for the best XGBoost model displayed.")